In [ ]:
#Test Benchmark. Used to explore concepts
import torch
import torch.distributions as dist

CUDA = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

P = 400
R = 4
C = 4

TRUE_ALPHA = dist.Normal(
                    loc = torch.zeros(R,C) + torch.eye(R,C),
                    scale = torch.ones(R,C) * 0.5
                ).sample().to(CUDA)
TRUE_DENSITY = torch.Tensor([1, 2, 1, 1.5]).to(CUDA)
TRUE_BETA = dist.Dirichlet( 
                    concentration = TRUE_ALPHA.exp() + TRUE_DENSITY[:, None]
                ).sample((P,))

POPULATION = dist.Poisson(rate = 4000).sample((P,)).to(CUDA)
X = dist.Dirichlet(
                    concentration = torch.ones(R)
                ).sample((P,)).to(CUDA)
X = (X * POPULATION[ : , None ]).ceil()
T = (X[: , :, None] * TRUE_BETA).sum(dim=1).round()

def display_transition_matrix(p_idx, TRUE_BETA, TRUE_ALPHA, X, T,  #Claude AI
                              
                               row_names=None, col_names=None):
    """
    Display the transition matrix for a single precinct.
    
    Shows beta[p,r,c] (observed fraction) and alpha[r,c] (concentration)
    with row sums (X) and column sums (T).
    
    Parameters
    ----------
    p_idx     : int          - precinct index to display
    TRUE_BETA : (P, R, C)   - transition probabilities
    TRUE_ALPHA: (R, C)       - Dirichlet concentrations (shared across P)
    X         : (P, R)       - group sizes per precinct
    T         : (P, C)       - vote counts per precinct
    row_names : list[str]    - group names,  defaults to ["Group 0", ...]
    col_names : list[str]    - party names,  defaults to ["Party 0", ...]
    """
    R, C = TRUE_ALPHA.shape

    row_names = row_names or [f"Group {r}" for r in range(R)]
    col_names = col_names or [f"Party {c}" for c in range(C)]

    beta  = TRUE_BETA[p_idx]          # (R, C)
    alpha = TRUE_ALPHA                 # (R, C)
    x_row = X[p_idx]                  # (R,)
    t_col = T[p_idx]                  # (C,)

    # ── Column widths ───────────────────────────────────────────────
    cell_w   = 18          # width of each data cell
    label_w  = 14          # width of the row-label column
    rowsum_w = 16          # width of the row-sum column

    # ── Header ──────────────────────────────────────────────────────
    col_header = "".join(f"{name:^{cell_w}}" for name in col_names)
    header = (f"\n  Transition Matrix — Precinct {p_idx}\n"
              f"  beta value  (alpha concentration)\n")
    divider = "─" * (label_w + cell_w * C + rowsum_w)

    print(header)
    print(f"{'':^{label_w}}{col_header}{'X (group N)':^{rowsum_w}}")
    print(divider)

    # ── Data rows ───────────────────────────────────────────────────
    for r in range(R):
        row_label = f"{row_names[r]:<{label_w}}"
        cells = ""
        for c in range(C):
            b = beta[r, c].item()
            a = alpha[r, c].item()
            cell = f"{b:.3f} ({a:.2f})"
            cells += f"{cell:^{cell_w}}"
        x_val = f"{x_row[r].item():>6.0f}"
        print(f"{row_label}{cells}{x_val:^{rowsum_w}}")

    print(divider)

    # ── Column sums (T) ─────────────────────────────────────────────
    col_sums = "".join(
        f"{t_col[c].item():^{cell_w}.0f}" for c in range(C)
    )
    total = t_col.sum().item()
    print(f"{'T (votes)':^{label_w}}{col_sums}{total:^{rowsum_w}.0f}")
    print()

display_transition_matrix(0, TRUE_BETA, TRUE_ALPHA, X, T)

ImportError: cannot import name 'nn' from partially initialized module 'torch' (most likely due to a circular import) (c:\Users\karli\anaconda3\envs\MCMC_py\Lib\site-packages\torch\__init__.py)

In [ ]:
from MCMC_paper_heteroprop import EI_MCMC

ALPHAS = EI_MCMC(X, T, 20, 100000, 20000, 0, (0,1), 0.5, 200, 0.3, 0.2, checkpoint=5000, save_betas=False)

ImportError: cannot import name 'nn' from partially initialized module 'torch' (most likely due to a circular import) (c:\Users\karli\anaconda3\envs\MCMC_py\Lib\site-packages\torch\__init__.py)

In [ ]:
import pickle

with open("ALPHAS.pkl", "wb") as f: pickle.dump(ALPHAS.cpu())
#with open("ALPHAS.pkl", "rb") as f: ALPHAS = pickle.load(f)

AttributeError: 'tuple' object has no attribute 'cpu'